# 5X72 experimental references: saved score and geometry diagnostic

The [prespecified workflow](5x72_reference_workflow.md) retains eight fixed reference evaluations and compares all 156 historical returned poses in both saved score contexts. This notebook reads saved evidence. It runs **no docking, scoring, optimization or preparation**. Heavy coordinates are experimental; H orientation is generated. P59 controls P69 within each ligand's own context. No cross-ligand affinity ranking or production default changes. #17/#6/#33 remain partial/open.

In [1]:
import gzip, json, sys
from pathlib import Path
checkout = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p/'devtools/qualify_5x72_reference.py').is_file())
sys.path.insert(0, str(checkout))
import numpy as np
import pandas as pd
import pyunitwizard as puw
import dockingmt as dmt
from devtools.qualify_5x72_reference import OUTPUT, sha, load_archives, compare_saved, score_components

payload = OUTPUT.read_bytes()
record = json.loads(gzip.decompress(payload))
checkpoint = json.loads(OUTPUT.with_name('checkpoint_2026-10-09.json').read_text())
assert sha(payload) == checkpoint['producer_archive']['gzip_sha256']
assert record['new_searches'] == 0 and record['new_fixed_evaluations'] == 8
print('Original gzip SHA256:', sha(payload))
print('Preserved provider source:', record['qualified_provider_commit'])
scores = []
for ligand, reference in record['references'].items():
    for order, contexts in reference['fixed_scores'].items():
        for context, pose in contexts.items():
            components = score_components(pose)
            scores.append({'ligand':ligand, 'order':order, 'score_receptor':context,
                'total_kcal_mol':components['fixed'], 'intra_kcal_mol':components['fixed.lig_intra']})
    assert reference['root_order_agrees_at_0_001_kcal_mol']
    assert all(v == 0 for differences in reference['root_order_component_difference_kcal_mol'].values() for v in differences.values())
display(pd.DataFrame(scores))

Original gzip SHA256: 8b53aac76447951737e0f477124fa11b074c8e0d6ae4244eac686968d471c1d3
Preserved provider source: 5bd893c85fe8d211663b2b1f865f5f1d2c382a90


,ligand,order,score_receptor,total_kcal_mol,intra_kcal_mol
0,p59,first_fixed,occupied,-10.534,-0.296
1,p59,first_fixed,sham,-9.661,-0.296
2,p59,native,occupied,-10.534,-0.296
3,p59,native,sham,-9.661,-0.296
4,p69,first_fixed,occupied,-10.499,0.559
5,p69,first_fixed,sham,-9.626,0.559
6,p69,native,occupied,-10.499,0.559
7,p69,native,sham,-9.626,0.559


The original prepared rigid fragments have different internal pair distances from the experimental reference. These differences persist under the two allowed torsion rotations, but do not establish failure at the 2.5 Å recovery threshold or explain the physical cause. The following recomputes all 552 saved pair distances independently from their captured coordinates.

In [2]:
geometry_rows = []
checked_pairs = 0
for ligand, reference in record['references'].items():
    geometry = reference['geometry']
    source = np.asarray(geometry['source_heavy_coordinates_angstrom'])
    experimental = np.asarray(geometry['reference_heavy_coordinates_angstrom'])
    assert len(geometry['pairs']) == 276
    assert sum(pair['heavy_bond'] for pair in geometry['pairs']) == 27
    for pair in geometry['pairs']:
        i,j = pair['source_atom_pair']
        a = np.linalg.norm(source[i]-source[j])
        b = np.linalg.norm(experimental[i]-experimental[j])
        np.testing.assert_allclose([a,b,b-a], [pair['prepared_distance_angstrom'],
            pair['reference_distance_angstrom'],pair['reference_minus_prepared_angstrom']],rtol=0,atol=1e-10)
        checked_pairs += 1
    for summary in geometry['summaries']:
        geometry_rows.append({'ligand':ligand, **summary})
assert checked_pairs == 552
print('552 indexed heavy-atom pair records verified; no fit or symmetry matching.')
display(pd.DataFrame(geometry_rows).drop(columns='heavy_atom_indices'))

552 indexed heavy-atom pair records verified; no fit or symmetry matching.


,ligand,fragment,max_absolute_distance_difference_angstrom,pairs,rms_distance_difference_angstrom
0,p59,all,2.347988,276,0.660520
1,p59,0,0.130392,21,0.058255
2,p59,1,0.185919,55,0.088882
3,p59,2,0.047697,15,0.022179
4,p69,all,2.498244,276,0.744599
5,p69,0,0.029128,21,0.014066
6,p69,1,0.311531,55,0.121869
7,p69,2,0.043178,15,0.019288


All 312 saved pose/context comparisons and all eight captured reference evaluations are checked below. Negative pose-minus-reference total favors the retained pose in that same context. Exact ties stay ties. The intra component and best-pose intra component are equal in these unchanged evaluations, so the intra sign alone does not explain total-score ordering. Fixed-score components are distinct from docking-energy components.

In [3]:
archives = load_archives()[0]
rows, cells, evaluations = [], [], 0
with puw.context(standard_units=['pm','fs','K','mole','dalton','e','kJ/mol','radians']):
    for ligand, reference in record['references'].items():
        for order, contexts in reference['fixed_scores'].items():
            original = dmt.DockingPose.from_dict(reference['poses'][order])
            assert original.rank is None and original.scores == {}
            for data in contexts.values():
                scored = dmt.DockingPose.from_dict(data)
                history = scored.metadata['scoring_history'][-1]
                assert dmt.verify_captured_inputs(history['backend_artifacts'])
                assert history['operation']=='score' and len(history['scores'])==8
                np.testing.assert_array_equal(puw.get_value(scored.coordinates,to_unit='angstrom'),
                    puw.get_value(original.coordinates,to_unit='angstrom'))
                evaluations += 1
        compared, summarized = compare_saved(ligand, archives[ligand],reference['fixed_scores'])
        rows.extend(compared); cells.extend(summarized)
assert evaluations == 8 and len(rows)==312 and len(cells)==96
assert rows == record['comparison_rows'] and cells == record['cell_summaries']
table = pd.DataFrame(rows)
table['pose_below_reference'] = [r['pose_minus_reference_kcal_mol']['fixed'] < 0 for r in rows]
display(table.groupby(['ligand','search_receptor','score_receptor']).agg(
    saved_poses=('original_rank','size'),below_reference=('pose_below_reference','sum')))
example = [r for r in rows if r['ligand']=='p69' and r['search_receptor']=='occupied'
    and r['score_receptor']=='occupied' and r['order']=='native' and r['seed']==42 and r['exhaustiveness']==8]
display(pd.DataFrame([{'original_rank':r['original_rank'],'heavy_rmsd_angstrom':r['heavy_rmsd_angstrom'],
    'occupied_fixed_total_kcal_mol':r['pose_fixed_components_kcal_mol']['fixed'],
    'pose_minus_reference_kcal_mol':r['pose_minus_reference_kcal_mol']['fixed']} for r in example]))
print('All 156 retained poses / 312 saved comparisons checked; no historical score was recalculated.')

saved_poses  below_reference
ligand search_receptor score_receptor                              
p59    occupied        occupied                 28                0
                       sham                     28                0
       sham            occupied                 50                0
                       sham                     50               48
p69    occupied        occupied                 29               10
                       sham                     29                0
       sham            occupied                 49                0
                       sham                     49               16

,original_rank,heavy_rmsd_angstrom,occupied_fixed_total_kcal_mol,pose_minus_reference_kcal_mol
0,1,5.528500,-11.000,-0.501
1,2,5.103313,-9.539,0.960
2,3,2.020196,-10.320,0.179


All 156 retained poses / 312 saved comparisons checked; no historical score was recalculated.
